# Baseline — DummyClassifier

Dataset: IBM Telco Customer Churn (~7.000 registros, 20 features, target `Churn`)

Estratégias:
- `most_frequent`: sempre prediz a classe majoritária (No Churn)
- `stratified`: prediz aleatoriamente respeitando a proporção do target

## 1. Setup

In [ ]:
from pathlib import Path

import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd
from sklearn.dummy import DummyClassifier
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    roc_auc_score,
)
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
TARGET = "Churn"
TEST_SIZE = 0.2
PROJECT_ROOT = Path.cwd().parent

## 2. Carregar e limpar

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "raw" / "WA_Fn-UseC_-Telco-Customer-Churn.csv"
df = pd.read_csv(DATA_PATH)

df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")
df[TARGET] = df[TARGET].map({"Yes": 1, "No": 0})
df = df.drop(columns=["customerID"])

print(f"Shape: {df.shape}")
print(f"Distribuição do target:\n{df[TARGET].value_counts(normalize=True)}")

## 3. Split estratificado

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_SEED, stratify=y,
)
print(f"Treino: {X_train.shape[0]} amostras | Teste: {X_test.shape[0]} amostras")
print(f"Proporção churn treino: {y_train.mean():.3f}")
print(f"Proporção churn teste:  {y_test.mean():.3f}")

## 4. Dummy — `most_frequent`

In [ ]:
dummy_mf = DummyClassifier(strategy="most_frequent", random_state=RANDOM_SEED)
dummy_mf.fit(X_train, y_train)

y_pred_mf = dummy_mf.predict(X_test)
y_prob_mf = dummy_mf.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_mf):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob_mf):.4f}")
print(f"\n{classification_report(y_test, y_pred_mf, target_names=['No Churn', 'Churn'], zero_division=0)}")

cm_mf = confusion_matrix(y_test, y_pred_mf)
ConfusionMatrixDisplay(cm_mf, display_labels=["No Churn", "Churn"]).plot(cmap="Oranges")

## 5. Dummy — `stratified`

In [ ]:
dummy_st = DummyClassifier(strategy="stratified", random_state=RANDOM_SEED)
dummy_st.fit(X_train, y_train)

y_pred_st = dummy_st.predict(X_test)
y_prob_st = dummy_st.predict_proba(X_test)[:, 1]

print(f"Accuracy:  {accuracy_score(y_test, y_pred_st):.4f}")
print(f"ROC AUC:   {roc_auc_score(y_test, y_prob_st):.4f}")
print(f"\n{classification_report(y_test, y_pred_st, target_names=['No Churn', 'Churn'], zero_division=0)}")

cm_st = confusion_matrix(y_test, y_pred_st)
ConfusionMatrixDisplay(cm_st, display_labels=["No Churn", "Churn"]).plot(cmap="Oranges")

## 6. Registrar no MLflow

In [ ]:
mlflow.set_tracking_uri(f"sqlite:///{PROJECT_ROOT / 'mlflow.db'}")
mlflow.set_experiment("churn-baselines")

for name, model, y_pred, y_prob in [
    ("most_frequent", dummy_mf, y_pred_mf, y_prob_mf),
    ("stratified", dummy_st, y_pred_st, y_prob_st),
]:
    cm = confusion_matrix(y_test, y_pred)
    metrics = {
        "accuracy": accuracy_score(y_test, y_pred),
        "roc_auc": roc_auc_score(y_test, y_prob),
    }
    if cm[1, 0] + cm[1, 1] > 0:
        metrics["recall"] = cm[1, 1] / (cm[1, 0] + cm[1, 1])
    if cm[0, 1] + cm[1, 1] > 0:
        metrics["precision"] = cm[1, 1] / (cm[0, 1] + cm[1, 1])

    with mlflow.start_run(run_name=f"dummy_{name}"):
        mlflow.log_params({
            "model": "DummyClassifier",
            "strategy": name,
            "random_seed": RANDOM_SEED,
            "test_size": TEST_SIZE,
        })
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, "model")
        print(f"dummy_{name} → run_id: {mlflow.active_run().info.run_id} | metrics: {metrics}")